## 1. Nạp tập dữ liệu sạch từ file CSV

In [12]:
import os
import re
import pandas as pd
import numpy as np

# 1. Đường dẫn file CSV sạch vừa lưu
csv_path = os.path.abspath('../../data/notebook/content_base/jobs_clean.csv')
if not os.path.exists(csv_path):
    csv_path = os.path.abspath('data/notebook/content_base/jobs_clean.csv')

# 2. Đọc file CSV
jobs = pd.read_csv(csv_path)
print(f" Đã nạp thành công: {jobs.shape[0]:,} dòng x {jobs.shape[1]} cột")
print("Danh sách các cột:", list(jobs.columns))
jobs.head(5)


 Đã nạp thành công: 3,938 dòng x 18 cột
Danh sách các cột: ['job_id', 'company_name', 'title', 'description_clean', 'required_skills', 'preferred_skills', 'experience_min', 'experience_max', 'location', 'city', 'salary_min', 'salary_max', 'employment_type', 'work_mode', 'deadline', 'is_active', 'job_url', 'company_logo_url']


,job_id,company_name,title,description_clean,required_skills,preferred_skills,experience_min,experience_max,location,city,salary_min,salary_max,employment_type,work_mode,deadline,is_active,job_url,company_logo_url
0,topcv_2136013,1 THÀNH VIÊN CÔNG NGHỆ GIÁO DỤC SOTATEK,Kỹ Sư Hạ Tầng Công Nghệ Thông Tin,"1. Phụ trách lắp đặt, cấu hình và bảo trì các ...","Linux, An Ninh Mạng",NaN,1.0,1.0,"KCN Phước Nam 2, Ninh Thuận cũ, Xã Thuận Nam, ...",Khác,NaN,40000000.0,Full-time,Onsite,2026-11-08,True,https://www.topcv.vn/viec-lam/ky-su-ha-tang-co...,https://static.topcv.vn/company_logos/l4PqvWTq...
1,topcv_1986323,1 THÀNH VIÊN ISOFHCARE,Chuyên Viên BA Nghiệp Vụ,Hỗ trợ Phân tích Yêu cầu: Làm việc dưới sự hướ...,"SQL, Jira, Confluence, Agile, Scrum",NaN,3.0,3.0,"Tòa nhà One Mount. 458 Minh Khai, Phường Hai B...",Hà Nội,NaN,25000000.0,Full-time,Onsite,2026-11-25,True,https://www.topcv.vn/viec-lam/chuyen-vien-ba-n...,https://static.topcv.vn/company_logos/cong-ty-...
2,topcv_2077790,1 THÀNH VIÊN ISOFHCARE,Nhân Viên Phân Tích Yêu Cầu Ứng Dụng (BA),"Business Analyst chịu trách nhiệm thu thập, ph...","Machine Learning, Agile, Scrum",NaN,3.0,3.0,"One Mount Building, 458 P. Minh Khai, Phường H...",Hà Nội,NaN,25000000.0,Full-time,Onsite,2026-10-22,False,https://www.topcv.vn/viec-lam/nhan-vien-phan-t...,https://static.topcv.vn/company_logos/cong-ty-...
3,topcv_2339964,1 THÀNH VIÊN TRUYỀN THÔNG YUM NETWORK,Nhân Viên Designer Thiết Kế Parttime,"Thiết kế, quản lý và phát triển về mặt mỹ thuậ...","Animation Design, Photoshop",NaN,1.0,1.0,"23 Duy Tân,Cầu Giấy, Phường Cầu Giấy, Hà Nội",Hà Nội,5000000.0,6000000.0,Part-time,Onsite,2026-11-23,True,https://www.topcv.vn/viec-lam/nhan-vien-design...,https://static.topcv.vn/company_logos/681d5577...
4,topcv_2334350,1C,Chuyên Viên Kinh Doanh Phần Mềm Quản Trị,"Chủ động tìm kiếm, tiếp cận và phát triển khác...",NaN,NaN,2.0,2.0,"Tầng 21 Tòa Century - 458 đường Minh Khai, Tim...",Hà Nội,NaN,NaN,Full-time,Onsite,2026-11-18,True,https://www.topcv.vn/viec-lam/chuyen-vien-kinh...,https://static.topcv.vn/company_logos/rP26CTKe...


## 2. Kiểm tra Cấu trúc dữ liệu & Kiểu dữ liệu (Data Types)

In [13]:
print("=== TỔNG QUAN KIỂU DỮ LIỆU & BỘ NHỚ ===")
jobs.info()


=== TỔNG QUAN KIỂU DỮ LIỆU & BỘ NHỚ ===
<class 'pandas.DataFrame'>
RangeIndex: 3938 entries, 0 to 3937
Data columns (total 18 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   job_id             3938 non-null   str    
 1   company_name       3938 non-null   str    
 2   title              3938 non-null   str    
 3   description_clean  3938 non-null   str    
 4   required_skills    3273 non-null   str    
 5   preferred_skills   0 non-null      float64
 6   experience_min     3938 non-null   float64
 7   experience_max     3938 non-null   float64
 8   location           3938 non-null   str    
 9   city               3938 non-null   str    
 10  salary_min         1548 non-null   float64
 11  salary_max         1767 non-null   float64
 12  employment_type    3938 non-null   str    
 13  work_mode          3938 non-null   str    
 14  deadline           3938 non-null   str    
 15  is_active          3938 non-null   bool   


## 3. Kiểm tra Độ đầy đủ dữ liệu (Completeness & Missing Rate)

In [14]:
total_rows = len(jobs)
missing_stats = []

for col in jobs.columns:
    null_count = jobs[col].isnull().sum()
    # Đếm thêm chuỗi rỗng nếu là kiểu object/string
    empty_count = (jobs[col].astype(str).str.strip() == '').sum() if jobs[col].dtype == 'object' else 0
    total_missing = null_count + empty_count
    missing_pct = (total_missing / total_rows) * 100
    valid_pct = 100.0 - missing_pct
    
    missing_stats.append({
        'Cột': col,
        'Số dòng thiếu': total_missing,
        'Tỷ lệ thiếu (%)': round(missing_pct, 2),
        'Độ đầy đủ (%)': round(valid_pct, 2)
    })

df_missing = pd.DataFrame(missing_stats).sort_values(by='Độ đầy đủ (%)', ascending=False)
display(df_missing)


,Cột,Số dòng thiếu,Tỷ lệ thiếu (%),Độ đầy đủ (%)
0,job_id,0,0.00,100.00
1,company_name,0,0.00,100.00
2,title,0,0.00,100.00
3,description_clean,0,0.00,100.00
7,experience_max,0,0.00,100.00
6,experience_min,0,0.00,100.00
8,location,0,0.00,100.00
9,city,0,0.00,100.00
12,employment_type,0,0.00,100.00
13,work_mode,0,0.00,100.00


## 4. Thống kê mô tả các trường số (Mức lương & Kinh nghiệm)

Quy đổi cột lương sang đơn vị **triệu VNĐ/tháng** để dễ quan sát phân bố mức lương tối thiểu và tối đa.

In [15]:
# Thống kê mô tả số năm kinh nghiệm
print("=== PHÂN BỐ SỐ NĂM KINH NGHIỆM (NĂM) ===")
display(jobs[['experience_min', 'experience_max']].describe().round(2))

# Thống kê mức lương theo đơn vị triệu VNĐ/tháng
print("\n=== PHÂN BỐ MỨC LƯƠNG (TRIỆU VNĐ/THÁNG) ===")
salary_summary = pd.DataFrame({
    'Lương tối thiểu (triệu VNĐ)': (jobs['salary_min'] / 1_000_000).describe(),
    'Lương tối đa (triệu VNĐ)': (jobs['salary_max'] / 1_000_000).describe()
}).round(2)
display(salary_summary)


=== PHÂN BỐ SỐ NĂM KINH NGHIỆM (NĂM) ===


,experience_min,experience_max
count,3938.00,3938.00
mean,2.35,2.35
std,1.99,1.99
min,0.00,0.00
25%,1.00,1.00
50%,2.00,2.00
75%,3.00,3.00
max,20.00,20.00



=== PHÂN BỐ MỨC LƯƠNG (TRIỆU VNĐ/THÁNG) ===


,Lương tối thiểu (triệu VNĐ),Lương tối đa (triệu VNĐ)
count,1548.00,1767.00
mean,21983.77,32851.13
std,620304.93,1043258.32
min,0.00,0.00
25%,10.00,11.00
50%,14.00,18.00
75%,20.00,30.00
max,20000000.00,40000000.00


## 5. Phân bố các trường phân loại (Categorical Distribution)

In [16]:
# 1. Phân bố theo thành phố
print("=== 1. PHÂN BỐ THEO THÀNH PHỐ (CITY) ===")
city_df = jobs['city'].value_counts(dropna=False).reset_index()
city_df.columns = ['Thành phố', 'Số lượng']
city_df['Tỷ lệ (%)'] = (city_df['Số lượng'] / len(jobs) * 100).round(2)
display(city_df)

# 2. Phân bố theo mô hình làm việc
print("\n=== 2. PHÂN BỐ THEO MÔ HÌNH LÀM VIỆC (WORK_MODE) ===")
wm_df = jobs['work_mode'].value_counts(dropna=False).reset_index()
wm_df.columns = ['Work Mode', 'Số lượng']
wm_df['Tỷ lệ (%)'] = (wm_df['Số lượng'] / len(jobs) * 100).round(2)
display(wm_df)

# 3. Phân bố theo hình thức làm việc
print("\n=== 3. PHÂN BỐ THEO HÌNH THỨC LÀM VIỆC (EMPLOYMENT_TYPE) ===")
emp_df = jobs['employment_type'].value_counts(dropna=False).reset_index()
emp_df.columns = ['Employment Type', 'Số lượng']
emp_df['Tỷ lệ (%)'] = (emp_df['Số lượng'] / len(jobs) * 100).round(2)
display(emp_df)

# 4. Phân bố theo trạng thái hiệu lực
print("\n=== 4. TRẠNG THÁI HIỆU LỰC (IS_ACTIVE) ===")
act_df = jobs['is_active'].value_counts(dropna=False).reset_index()
act_df.columns = ['Is Active', 'Số lượng']
act_df['Tỷ lệ (%)'] = (act_df['Số lượng'] / len(jobs) * 100).round(2)
display(act_df)


=== 1. PHÂN BỐ THEO THÀNH PHỐ (CITY) ===


,Thành phố,Số lượng,Tỷ lệ (%)
0,Hà Nội,2094,53.17
1,Hồ Chí Minh,1079,27.40
2,Khác,699,17.75
3,Đà Nẵng,66,1.68



=== 2. PHÂN BỐ THEO MÔ HÌNH LÀM VIỆC (WORK_MODE) ===


,Work Mode,Số lượng,Tỷ lệ (%)
0,Onsite,3891,98.81
1,Remote,32,0.81
2,Hybrid,15,0.38



=== 3. PHÂN BỐ THEO HÌNH THỨC LÀM VIỆC (EMPLOYMENT_TYPE) ===


,Employment Type,Số lượng,Tỷ lệ (%)
0,Full-time,3756,95.38
1,Intern,163,4.14
2,Contract,15,0.38
3,Part-time,4,0.10



=== 4. TRẠNG THÁI HIỆU LỰC (IS_ACTIVE) ===


,Is Active,Số lượng,Tỷ lệ (%)
0,True,3140,79.74
1,False,798,20.26


## 6. Phân tích đặc trưng văn bản & Top Kỹ năng yêu cầu

In [17]:
# 1. Thống kê độ dài văn bản mô tả công việc (description_clean)
desc_lengths = jobs['description_clean'].dropna().str.len()
print("=== ĐỘ DÀI MÔ TẢ CÔNG VIỆC (SỐ KÝ TỰ) ===")
display(desc_lengths.describe().round(0))

# 2. Bóc tách và đếm tần suất Top 20 kỹ năng yêu cầu
from collections import Counter

skill_counter = Counter()
for text in jobs['required_skills'].dropna():
    skills = [s.strip() for s in str(text).split(',') if s.strip()]
    skill_counter.update(skills)

top_skills_df = pd.DataFrame(skill_counter.most_common(20), columns=['Kỹ năng', 'Số lần xuất hiện'])
top_skills_df['Tỷ lệ (%)'] = (top_skills_df['Số lần xuất hiện'] / len(jobs) * 100).round(2)

print("\n=== TOP 20 KỸ NĂNG XUẤT HIỆN NHIỀU NHẤT TRONG DATASET ===")
display(top_skills_df)


=== ĐỘ DÀI MÔ TẢ CÔNG VIỆC (SỐ KÝ TỰ) ===


count     3938.0
mean      2359.0
std       1171.0
min        340.0
25%       1570.0
50%       2124.0
75%       2841.0
max      12642.0
Name: description_clean, dtype: float64


=== TOP 20 KỸ NĂNG XUẤT HIỆN NHIỀU NHẤT TRONG DATASET ===


,Kỹ năng,Số lần xuất hiện,Tỷ lệ (%)
0,SQL,545,13.84
1,Python,456,11.58
2,Photoshop,386,9.80
3,Agile,378,9.60
4,Git,304,7.72
5,CI/CD,276,7.01
6,Scrum,269,6.83
7,Java,265,6.73
8,Excel,260,6.60
9,Linux,258,6.55


## 6. Xây dựng "Content Soup" (Tổ hợp đặc trưng có gán trọng số)

Để ưu tiên các trường quan trọng nhất trong việc ra quyết định tuyển dụng:
- **Chức danh (`title`)**: Trọng số x3 (xác định định hướng vị trí công việc).
- **Kỹ năng cốt lõi (`required_skills`)**: Trọng số x4 (yếu tố cốt lõi quyết định độ khớp).
- **Mô tả chi tiết (`description_clean`)**: Trọng số x1 (cung cấp thêm ngữ cảnh về dự án và công nghệ).

In [18]:
def build_content_soup(row):
    title = str(row.get('title', '')) if pd.notnull(row.get('title')) else ''
    skills = str(row.get('required_skills', '')) if pd.notnull(row.get('required_skills')) else ''
    desc = str(row.get('description_clean', '')) if pd.notnull(row.get('description_clean')) else ''
    
    # Nhân trọng số bằng cách lặp lại từ khóa
    soup = f"{title} {title} {title} {skills} {skills} {skills} {skills} {desc}"
    return soup.strip()

jobs['content_soup'] = jobs.apply(build_content_soup, axis=1)

print("=== MẪU CONTENT SOUP CỦA 2 CÔNG VIỆC ĐẦU TIÊN ===")
for idx, s in enumerate(jobs['content_soup'].head(2)):
    print(f"\n[Job {idx+1}] {jobs.iloc[idx]['title']}:")
    print(s[:300] + "...")


=== MẪU CONTENT SOUP CỦA 2 CÔNG VIỆC ĐẦU TIÊN ===

[Job 1] Kỹ Sư Hạ Tầng Công Nghệ Thông Tin:
Kỹ Sư Hạ Tầng Công Nghệ Thông Tin Kỹ Sư Hạ Tầng Công Nghệ Thông Tin Kỹ Sư Hạ Tầng Công Nghệ Thông Tin Linux, An Ninh Mạng Linux, An Ninh Mạng Linux, An Ninh Mạng Linux, An Ninh Mạng 1. Phụ trách lắp đặt, cấu hình và bảo trì các thiết bị phần cứng IT như máy tính, máy in, máy chủ, hệ thống an ninh…, ...

[Job 2] Chuyên Viên BA Nghiệp Vụ:
Chuyên Viên BA Nghiệp Vụ Chuyên Viên BA Nghiệp Vụ Chuyên Viên BA Nghiệp Vụ SQL, Jira, Confluence, Agile, Scrum SQL, Jira, Confluence, Agile, Scrum SQL, Jira, Confluence, Agile, Scrum SQL, Jira, Confluence, Agile, Scrum Hỗ trợ Phân tích Yêu cầu: Làm việc dưới sự hướng dẫn của Senior BA để thu thập, g...


## 7. Vector hóa với TF-IDF & Tính Ma trận Cosine Similarity

- Sử dụng `TfidfVectorizer` với `ngram_range=(1, 2)` để nắm bắt cả từ đơn (`python`, `sql`) và cụm từ ghép (`spring boot`, `machine learning`, `data analyst`).
- Giới hạn `max_features=10,000` và `min_df=2` để lọc bớt từ hiếm xuất hiện dưới 2 lần, tối ưu bộ nhớ.

In [19]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import time

start_time = time.time()

# 1. Khởi tạo và fit-transform TfidfVectorizer
tfidf = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1, 2),
    min_df=2
)
tfidf_matrix = tfidf.fit_transform(jobs['content_soup'])

# 2. Tính ma trận Cosine Similarity giữa tất cả các công việc (Job-to-Job)
cosine_sim_matrix = cosine_similarity(tfidf_matrix, tfidf_matrix)

elapsed = time.time() - start_time
print(f" Đã vector hóa thành công {tfidf_matrix.shape[0]:,} jobs x {tfidf_matrix.shape[1]:,} features")
print(f" Ma trận Cosine Similarity: {cosine_sim_matrix.shape} (Dung lượng: ~{cosine_sim_matrix.nbytes / 1024**2:.1f} MB)")
print(f"⏱ Thời gian thực thi: {elapsed:.2f} giây")


 Đã vector hóa thành công 3,938 jobs x 10,000 features
 Ma trận Cosine Similarity: (3938, 3938) (Dung lượng: ~118.3 MB)
⏱ Thời gian thực thi: 5.08 giây


## 9. Hệ thống Gợi ý Việc làm Toàn diện theo Hồ sơ & CV Ứng viên (Unified Recommender Engine)

Hệ thống hợp nhất linh hoạt cho phép ứng viên cung cấp thông tin theo nhiều cách:
1. **Tự chọn kỹ năng & Chức danh** (nhập danh sách `skills`).
2. **Dán đoạn văn bản CV / Tóm tắt năng lực** (`cv_text`): Hệ thống tự động bóc tách kỹ năng công nghệ (`extract_skills_from_text`).
3. **Kết hợp cả hai**: Dán CV + tích chọn thêm kỹ năng + đặt bộ lọc lương tối thiểu (`min_salary`), địa điểm (`preferred_city`), kinh nghiệm (`user_exp`).
4. **Phân tích khoảng cách kỹ năng (Skill Gap Analysis)**: Chỉ rõ những gì bạn đã đáp ứng (Điểm mạnh) và những gì công ty cần mà bạn còn thiếu (Gap cần học thêm).
5. **So khớp chuyên sâu 1:1 (`analyze_cv_fit_for_job`)**: Báo cáo phân tích chi tiết hồ sơ với một công việc cụ thể.

In [20]:
import os
import sys
import re
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# Tích hợp bộ danh bạ quy chuẩn kỹ năng (Canonical Skill Map) từ src/data_cleaning
try:
    sys.path.append(os.path.abspath('../..'))
    from src.data_cleaning.skill_canonical import CANONICAL_SKILL_MAP, extract_skills_from_text
except ImportError:
    # Dự phòng nếu chạy độc lập mà không import được module
    CANONICAL_SKILL_MAP = {
        'python': 'Python', 'java': 'Java', 'javascript': 'JavaScript', 'js': 'JavaScript',
        'typescript': 'TypeScript', 'ts': 'TypeScript', 'c++': 'C++', 'c#': 'C#',
        'php': 'PHP', 'golang': 'Golang', 'go': 'Golang', 'ruby': 'Ruby', 'rust': 'Rust',
        'react': 'React', 'reactjs': 'React', 'vue': 'Vue', 'vuejs': 'Vue', 'angular': 'Angular',
        'node': 'Node.js', 'nodejs': 'Node.js', 'spring boot': 'Spring Boot', 'springboot': 'Spring Boot',
        'laravel': 'Laravel', '.net': '.NET', 'dotnet': '.NET', 'sql': 'SQL', 'mysql': 'MySQL',
        'postgresql': 'PostgreSQL', 'mongodb': 'MongoDB', 'redis': 'Redis', 'kafka': 'Kafka',
        'docker': 'Docker', 'kubernetes': 'Kubernetes', 'k8s': 'Kubernetes', 'aws': 'AWS',
        'azure': 'Azure', 'gcp': 'GCP', 'ci/cd': 'CI/CD', 'git': 'Git', 'linux': 'Linux',
        'power bi': 'Power BI', 'tableau': 'Tableau', 'excel': 'Excel', 'spark': 'Spark'
    }
    def extract_skills_from_text(text):
        text_lower = str(text).lower()
        detected = []
        seen = set()
        for key, val in CANONICAL_SKILL_MAP.items():
            if re.search(r'\b' + re.escape(key) + r'\b', text_lower):
                if val.lower() not in seen:
                    seen.add(val.lower())
                    detected.append(val)
        return detected

def format_salary_display(s_min, s_max):
    """Chuẩn hoá hiển thị mức lương (tránh lỗi 15 - 15 tr khi min == max)."""
    if pd.notnull(s_min) and pd.notnull(s_max):
        if s_min == s_max:
            return f"{s_min/1e6:.0f} tr"
        return f"{s_min/1e6:.0f} - {s_max/1e6:.0f} tr"
    elif pd.notnull(s_max):
        return f"Lên đến {s_max/1e6:.0f} tr"
    elif pd.notnull(s_min):
        return f"Từ {s_min/1e6:.0f} tr"
    return "Thoả thuận"

def recommend_jobs(target_title="", cv_text="", skills=None, user_exp=1.0, preferred_city="Tất cả", min_salary=None, top_n=10):
    """
    Hàm gợi ý việc làm toàn diện kết hợp: Vị trí + Kỹ năng chọn tay + Đoạn văn bản CV + Bộ lọc kinh nghiệm/lương/địa điểm.
    """
    manual_skills = skills or []
    extracted_skills = extract_skills_from_text(cv_text) if cv_text else []
    
    # 1. Hợp nhất kỹ năng (loại bỏ trùng lặp)
    all_skills = []
    seen_skills = set()
    for s in manual_skills + extracted_skills:
        if s.lower() not in seen_skills:
            seen_skills.add(s.lower())
            all_skills.append(s)
            
    skills_soup = " ".join(all_skills)
    
    # 2. Xây dựng Profile Soup có gán trọng số (3x Title + 4x Skills + 1x CV Text)
    parts = []
    if target_title:
        parts.extend([target_title, target_title, target_title])
    if skills_soup:
        parts.extend([skills_soup, skills_soup, skills_soup, skills_soup])
    if cv_text:
        parts.append(cv_text)
        
    profile_soup = " ".join(parts).strip()
    if not profile_soup:
        print(" Vui lòng cung cấp ít nhất một thông tin: Vị trí (target_title), Kỹ năng (skills), hoặc Văn bản CV (cv_text).")
        return pd.DataFrame()
        
    # 3. Vectorize và tính Cosine Similarity (Stage 1)
    profile_vec = tfidf.transform([profile_soup])
    scores = cosine_similarity(profile_vec, tfidf_matrix).flatten()
    
    # 4. Lọc nghiệp vụ & Phân tích khoảng cách kỹ năng (Stage 2)
    results = []
    for idx, score in enumerate(scores):
        row = jobs.iloc[idx]
        if 'is_active' in row and not row['is_active']:
            continue
        if preferred_city != 'Tất cả':
            is_remote = bool(row.get('is_remote', False)) or str(row.get('work_mode', '')).lower() == 'remote'
            if row.get('city') != preferred_city and not is_remote:
                continue
        job_min_exp = row.get('experience_min')
        if pd.notnull(job_min_exp) and job_min_exp > (user_exp + 1.0):
            continue
        if min_salary is not None and pd.notnull(row.get('salary_max')) and row['salary_max'] < min_salary:
            continue
            
        job_skills = [s.strip() for s in str(row.get('required_skills', '')).split(',') if s.strip()]
        job_skills_lower = [s.lower() for s in job_skills]
        
        matched = [s for s, sl in zip(job_skills, job_skills_lower) if sl in seen_skills]
        missing = [s for s, sl in zip(job_skills, job_skills_lower) if sl not in seen_skills]
        coverage_pct = round(len(matched) / len(job_skills) * 100, 1) if job_skills else 100.0
        
        results.append({
            'Tiêu đề': row['title'],
            'Công ty': row['company_name'],
            'Độ phù hợp (%)': round(score * 100, 1),
            'Độ phủ kỹ năng (%)': f"{coverage_pct}% ({len(matched)}/{len(job_skills)})",
            'Kỹ năng đã có (Điểm mạnh)': ', '.join(matched[:4]) if matched else 'Khớp theo vị trí/mô tả',
            'Kỹ năng còn thiếu (Gap)': ', '.join(missing[:4]) if missing else 'Đầy đủ',
            'Kinh nghiệm yêu cầu': f"{row['experience_min']} năm" if pd.notnull(row.get('experience_min')) else 'Không yêu cầu',
            'Thành phố': row.get('city', 'Không rõ'),
            'Mức lương': format_salary_display(row.get('salary_min'), row.get('salary_max')),
            'job_url': row.get('job_url', ''),
            'company_logo_url': row.get('company_logo_url', '')
        })
        
    df_res = pd.DataFrame(results).sort_values(by='Độ phù hợp (%)', ascending=False)
    return df_res.head(top_n)

def analyze_cv_fit_for_job(job_identifier, cv_text="", skills=None, user_exp=1.0):
    """
    Báo cáo đánh giá chi tiết 1:1 giữa hồ sơ ứng viên và một công việc cụ thể.
    """
    if isinstance(job_identifier, (int, np.integer)):
        idx = int(job_identifier)
    elif isinstance(job_identifier, str):
        match = jobs[jobs['job_id'] == job_identifier]
        if match.empty:
            match = jobs[jobs['title'].str.contains(job_identifier, case=False, na=False, regex=False)]
        if match.empty:
            return f" Không tìm thấy công việc: {job_identifier}"
        idx = match.index[0]
        
    job = jobs.iloc[idx]
    
    manual_skills = skills or []
    extracted_skills = extract_skills_from_text(cv_text) if cv_text else []
    all_user_skills = []
    seen_skills = set()
    for s in manual_skills + extracted_skills:
        if s.lower() not in seen_skills:
            seen_skills.add(s.lower())
            all_user_skills.append(s)
            
    input_text = f"{' '.join(all_user_skills)} {cv_text}".strip()
    user_vec = tfidf.transform([input_text])
    job_vec = tfidf.transform([job['content_soup']])
    sim_score = cosine_similarity(user_vec, job_vec)[0][0]
    
    job_skills = [s.strip() for s in str(job.get('required_skills', '')).split(',') if s.strip()]
    job_skills_lower = [s.lower() for s in job_skills]
    
    matched = [s for s, sl in zip(job_skills, job_skills_lower) if sl in seen_skills]
    missing = [s for s, sl in zip(job_skills, job_skills_lower) if sl not in seen_skills]
    
    exp_required = job.get('experience_min') if pd.notnull(job.get('experience_min')) else 0.0
    if user_exp >= exp_required:
        exp_status = f" Đạt yêu cầu ({user_exp} năm >= {exp_required} năm)"
    elif user_exp + 1.0 >= exp_required:
        exp_status = f" Chấp nhận được ({user_exp} năm xấp xỉ yêu cầu {exp_required} năm)"
    else:
        exp_status = f" Chưa đủ kinh nghiệm ({user_exp} năm < yêu cầu {exp_required} năm)"
        
    coverage_pct = round(len(matched) / len(job_skills) * 100, 1) if job_skills else 100.0
    
    report = f"""================================================================================
 BÁO CÁO ĐÁNH GIÁ ĐỘ PHÙ HỢP HỒ SƠ VỚI CÔNG VIỆC (FIT REPORT)
================================================================================
Công việc mục tiêu: {job['title']}
Công ty:             {job['company_name']}
Địa điểm:            {job.get('city', 'Không rõ')} ({job.get('work_mode', 'Tại văn phòng')})
Mức lương:           {format_salary_display(job.get('salary_min'), job.get('salary_max'))}

--- 1. ĐIỂM SỐ TỔNG QUAN ---
• Điểm tương đồng nội dung (Content Similarity): {sim_score * 100:.1f}%
• Tỷ lệ đáp ứng kỹ năng (Skill Coverage):         {coverage_pct}% ({len(matched)}/{len(job_skills)} kỹ năng)
• Đánh giá số năm kinh nghiệm:                   {exp_status}

--- 2. CHI TIẾT KỸ NĂNG ---
 Kỹ năng bạn đã có ({len(matched)}):
   {', '.join(matched) if matched else 'Không có kỹ năng trùng khớp trực tiếp'}
 Kỹ năng còn thiếu mà công ty yêu cầu ({len(missing)}):
   {', '.join(missing) if missing else 'Tuyệt vời! Bạn đáp ứng đầy đủ tất cả kỹ năng cốt lõi'}

--- 3. LỜI KHUYÊN DÀNH CHO BẠN ---
"""
    if missing:
        report += f" Để tăng cơ hội trúng tuyển, bạn nên bổ sung thêm kinh nghiệm hoặc học thêm về: {', '.join(missing)}.\n"
    else:
        report += " Hồ sơ của bạn rất mạnh cho vị trí này! Hãy tự tin nộp đơn ứng tuyển ngay.\n"
    report += "================================================================================"
    return report

print(" Đã cập nhật thành công Recommender Engine đồng bộ với script .py:")
print("- Danh bạ kỹ năng mở rộng (Canonical Skill Map)")
print("- Chuẩn hoá hiển thị mức lương (tránh lỗi 15 - 15 tr)")
print("- Bổ sung trường job_url & company_logo_url phục vụ giao diện ứng tuyển")


 Đã cập nhật thành công Recommender Engine đồng bộ với script .py:
- Danh bạ kỹ năng mở rộng (Canonical Skill Map)
- Chuẩn hoá hiển thị mức lương (tránh lỗi 15 - 15 tr)
- Bổ sung trường job_url & company_logo_url phục vụ giao diện ứng tuyển


## 10. Thử nghiệm Thực tế với các Kịch bản Đa dạng (Case Studies)

In [21]:
# --- Case 1: Ứng viên chỉ chọn kỹ năng và vị trí mong muốn ---
print("=== CASE 1: ỨNG VIÊN CHỈ CHỌN KỸ NĂNG BẰNG TAY (Backend Python, Hà Nội) ===")
res1 = recommend_jobs(
    target_title="Backend Developer Python",
    skills=["Python", "SQL", "Docker", "PostgreSQL"],
    user_exp=1.0,
    preferred_city="Hà Nội",
    top_n=5
)
display(res1)

# --- Case 2: Ứng viên dán đoạn mô tả kinh nghiệm / CV ---
print("\n=== CASE 2: ỨNG VIÊN DÁN VĂN BẢN CV (DevOps / Microservices, Hồ Chí Minh) ===")
cv_devops = """
Kỹ sư DevOps với 3 năm kinh nghiệm làm việc với hệ thống Microservices.
Thành thạo Docker, Kubernetes (K8s), xây dựng CI/CD pipeline tự động hóa với GitLab CI và Jenkins.
Kinh nghiệm quản trị hệ thống Linux, triển khai hạ tầng đám mây trên AWS (EC2, S3, RDS, EKS).
"""
res2 = recommend_jobs(
    target_title="DevOps Engineer",
    cv_text=cv_devops,
    user_exp=3.0,
    preferred_city="Hồ Chí Minh",
    top_n=5
)
display(res2)

# --- Case 3: Kết hợp cả CV + Kỹ năng chọn thêm + Lọc lương tối thiểu ---
print("\n=== CASE 3: KẾT HỢP CV + KỸ NĂNG BỔ SUNG + LỌC LƯƠNG TỐI THIỂU >= 12 TRIỆU ===")
cv_da = """
Chuyên viên phân tích dữ liệu với 1 năm kinh nghiệm.
Sử dụng thành thạo SQL để truy vấn và xử lý dữ liệu lớn, thành thạo Excel và Power BI xây dựng Dashboard quản trị kinh doanh.
"""
res3 = recommend_jobs(
    target_title="Data Analyst",
    cv_text=cv_da,
    skills=["Python", "Tableau"],
    user_exp=1.0,
    preferred_city="Tất cả",
    min_salary=12000000,
    top_n=5
)
display(res3)

# --- Case 4: So khớp chuyên sâu 1:1 giữa CV DevOps và công việc gợi ý Top 1 ---
top1_job = res2.iloc[0]['Tiêu đề']
rep4 = analyze_cv_fit_for_job(job_identifier=top1_job, cv_text=cv_devops, user_exp=3.0)
print("\n" + rep4)


=== CASE 1: ỨNG VIÊN CHỈ CHỌN KỸ NĂNG BẰNG TAY (Backend Python, Hà Nội) ===


,Tiêu đề,Công ty,Độ phù hợp (%),Độ phủ kỹ năng (%),Kỹ năng đã có (Điểm mạnh),Kỹ năng còn thiếu (Gap),Kinh nghiệm yêu cầu,Thành phố,Mức lương,job_url,company_logo_url
906,Senior Backend Developer (Python),VODAPLAY,21.0,19.0% (4/21),"Python, SQL, PostgreSQL, Docker","An Ninh Mạng, R, Bash, Apache Kafka",2.0 năm,Hà Nội,Thoả thuận,https://www.topcv.vn/viec-lam/senior-backend-d...,https://static.topcv.vn/company_logos/cong-ty-...
460,Middle Data Analyst (Từ 2 Năm Kinh Nghiệm),Khoa học Dữ liệu,20.8,40.0% (2/5),"Python, SQL","Power BI, Excel, Metabase",2.0 năm,Hà Nội,15 tr,https://www.topcv.vn/viec-lam/middle-data-anal...,https://static.topcv.vn/company_logos/cong-ty-...
42,SOC Analyst Tier 2,An Ninh An toàn Thông tin CMC,20.4,66.7% (2/3),"Python, SQL",Linux,2.0 năm,Hà Nội,Thoả thuận,https://www.topcv.vn/viec-lam/soc-analyst-tier...,https://static.topcv.vn/company_logos/wADV14D3...
369,IT System (Lead) - Onboard Sớm - Part Time,Giải pháp Phân tích dữ liệu Insight Data,19.8,42.9% (3/7),"Python, SQL, Docker","It Helpdesk, It Support, Excel, Linux",1.0 năm,Hà Nội,Thoả thuận,https://www.topcv.vn/viec-lam/it-system-lead-o...,https://static.topcv.vn/company_logos/J76h2Bmn...
294,Backend Developer,EAERA,19.4,22.2% (4/18),"Python, SQL, PostgreSQL, Docker","JavaScript, TypeScript, Ruby, Apache Kafka",1.0 năm,Hà Nội,400 tr,https://www.topcv.vn/viec-lam/backend-develope...,https://static.topcv.vn/company_logos/cong-ty-...



=== CASE 2: ỨNG VIÊN DÁN VĂN BẢN CV (DevOps / Microservices, Hồ Chí Minh) ===


,Tiêu đề,Công ty,Độ phù hợp (%),Độ phủ kỹ năng (%),Kỹ năng đã có (Điểm mạnh),Kỹ năng còn thiếu (Gap),Kinh nghiệm yêu cầu,Thành phố,Mức lương,job_url,company_logo_url
379,Devops Engineer,IOTLINK,50.5,38.9% (7/18),"AWS, Docker, Kubernetes, CI/CD","Python, JavaScript, PHP, Ruby",1.0 năm,Hồ Chí Minh,20 tr,https://www.topcv.vn/viec-lam/devops-engineer/...,https://static.topcv.vn/company_logos/cong-ty-...
99,Senior DevOps Engineer,Chứng khoán HD,38.5,33.3% (6/18),"Docker, Kubernetes, CI/CD, Git","Apache Kafka, PostgreSQL, MySQL, MongoDB",4.0 năm,Hồ Chí Minh,Thoả thuận,https://www.topcv.vn/viec-lam/senior-devops-en...,https://static.topcv.vn/company_logos/pT90StgQ...
335,DevOps Engineer,Globaltech,33.7,41.2% (7/17),"Docker, Microservices, Kubernetes, CI/CD","Devops, Ci, Cd, Python",1.0 năm,Hồ Chí Minh,Thoả thuận,https://topdev.vn/viec-lam/devops-engineer-glo...,https://assets.saramin.vn/images/2022/03/11/To...
581,NodeJS Backend (TeamLead),TECHVIFY SOFTWARE.,33.6,37.5% (6/16),"Microservices, AWS, Docker, Kubernetes","Node.js, NestJS, GraphQL, gRPC",2.0 năm,Hồ Chí Minh,Thoả thuận,https://www.topcv.vn/viec-lam/nodejs-backend-t...,https://static.topcv.vn/company_logos/683687a5...
715,System Engineer (Data Center),Viễn Thông Quốc Tế FPT,31.3,55.6% (5/9),"AWS, Docker, Kubernetes, CI/CD","Python, PHP, Bash, Azure",2.0 năm,Hồ Chí Minh,Thoả thuận,https://www.topcv.vn/viec-lam/system-engineer-...,https://static.topcv.vn/company_logos/cong-ty-...



=== CASE 3: KẾT HỢP CV + KỸ NĂNG BỔ SUNG + LỌC LƯƠNG TỐI THIỂU >= 12 TRIỆU ===


,Tiêu đề,Công ty,Độ phù hợp (%),Độ phủ kỹ năng (%),Kỹ năng đã có (Điểm mạnh),Kỹ năng còn thiếu (Gap),Kinh nghiệm yêu cầu,Thành phố,Mức lương,job_url,company_logo_url
760,Middle Data Analyst (Từ 2 Năm Kinh Nghiệm),Khoa học Dữ liệu,47.9,80.0% (4/5),"Python, SQL, Power BI, Excel",Metabase,2.0 năm,Hà Nội,15 tr,https://www.topcv.vn/viec-lam/middle-data-anal...,https://static.topcv.vn/company_logos/cong-ty-...
975,Nhân Viên Phân Tích Giữ Liệu & Dashboard Tại G...,Nông Nghiệp BaF,47.6,50.0% (3/6),"SQL, Power BI, Excel","DAX, Power Query, Data Modeling",1.0 năm,Hà Nội,22 - 27 tr,https://www.topcv.vn/viec-lam/nhan-vien-phan-t...,https://static.topcv.vn/company_logos/6WYO6IwO...
551,Data Analyst (Mảng Tài Chính),GALAXY DEBT TRADING COMPANY LIMITED,47.2,100.0% (4/4),"Python, SQL, Power BI, Excel",Đầy đủ,1.0 năm,Hồ Chí Minh,15 - 20 tr,https://www.topcv.vn/brand/galaxydebt/tuyen-du...,https://static.topcv.vn/company_logos/galaxy-d...
973,Chuyên Viên Phân Tích Dữ Liệu,Nhựa Tái Chế Duy Tân,46.8,37.5% (3/8),"SQL, Power BI, Excel","System Administration, Technical Support, Data...",2.0 năm,Khác,13 - 15 tr,https://www.vietnamworks.com/chuyen-vien-phan-...,https://images.vietnamworks.com/pictureofcompa...
1277,Business Planning Analyst,Trung Tâm Anh Ngữ ILA,44.4,100.0% (3/3),"SQL, Power BI, Excel",Đầy đủ,2.0 năm,Hồ Chí Minh,Thoả thuận,https://www.topcv.vn/viec-lam/business-plannin...,https://static.topcv.vn/company_logos/69c604d2...



 BÁO CÁO ĐÁNH GIÁ ĐỘ PHÙ HỢP HỒ SƠ VỚI CÔNG VIỆC (FIT REPORT)
Công việc mục tiêu: Senior Devops Engineer
Công ty:             Aggregatori Capaci
Địa điểm:            Hà Nội (Onsite)
Mức lương:           Thoả thuận

--- 1. ĐIỂM SỐ TỔNG QUAN ---
• Điểm tương đồng nội dung (Content Similarity): 29.6%
• Tỷ lệ đáp ứng kỹ năng (Skill Coverage):         58.3% (7/12 kỹ năng)
• Đánh giá số năm kinh nghiệm:                    Chấp nhận được (3.0 năm xấp xỉ yêu cầu 4.0 năm)

--- 2. CHI TIẾT KỸ NĂNG ---
 Kỹ năng bạn đã có (7):
   Microservices, AWS, Docker, Kubernetes, CI/CD, Linux, Jenkins
 Kỹ năng còn thiếu mà công ty yêu cầu (5):
   GCP, Terraform, Nginx, Prometheus, Grafana

--- 3. LỜI KHUYÊN DÀNH CHO BẠN ---
 Để tăng cơ hội trúng tuyển, bạn nên bổ sung thêm kinh nghiệm hoặc học thêm về: GCP, Terraform, Nginx, Prometheus, Grafana.


## 11. Đánh giá & Đo lường Hiệu năng Hệ thống Gợi ý (Evaluation Metrics for CBF)

Đối với hệ thống gợi ý dựa trên nội dung (Content-Based Filtering) khi **chưa có log tương tác người dùng thực tế** (Clicks / Applies / Bookmark), ta đánh giá khách quan thông qua **5 nhóm chỉ số chuẩn của RecSys**:

1. **Skill Coverage Rate (SCR@K)**: Tỷ lệ phần trăm các kỹ năng công việc yêu cầu mà ứng viên thực sự đáp ứng được trong top K việc làm được gợi ý.
2. **Intra-List Diversity (ILD@K)**: Đo lường độ đa dạng bên trong danh sách top K gợi ý thông qua khoảng cách Cosine Distance giữa các cặp công việc:
   $$\text{ILD@K} = \frac{2}{K(K-1)} \sum_{i=1}^{K} \sum_{j=i+1}^{K} (1 - \text{CosineSimilarity}(\vec{v}_i, \vec{v}_j))$$
   *Ý nghĩa:* ILD cao chứng tỏ hệ thống gợi ý đa dạng phong phú, tránh tạo ra "bong bóng thông tin" (filter bubble) bị trùng lặp công việc.
3. **Domain Relevance (Precision@K)**: Tỷ lệ công việc trong top K thuộc đúng chuyên ngành/lĩnh vực mục tiêu của ứng viên (Backend, Frontend, Data, DevOps, Mobile, QA).
4. **Catalog Coverage**: Tỷ lệ phần trăm số lượng tin tuyển dụng trong toàn bộ kho dữ liệu (3,938 jobs) được gợi ý tới người dùng khi quét qua tập benchmark các phân hệ IT khác nhau.
5. **Cosine Similarity Distribution**: Phân tích phổ điểm số tương đồng nội dung (Mean, Std, Min, Max, Median) để kiểm chứng độ nhạy và phân định rõ ràng của không gian vector TF-IDF.

In [22]:
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
import matplotlib.pyplot as plt

# 1. Khởi tạo tập Benchmark Test Profiles đại diện cho 5 phân hệ IT chính (25 profiles)
test_benchmark_profiles = [
    # Phân hệ Backend (5 profiles)
    {"domain": "Backend", "title": "Backend Python Developer", "skills": ["Python", "Django", "FastAPI", "PostgreSQL", "Docker", "Redis"], "exp": 2.0, "city": "Hà Nội"},
    {"domain": "Backend", "title": "Java Spring Boot Engineer", "skills": ["Java", "Spring Boot", "MySQL", "Microservices", "Kafka"], "exp": 3.0, "city": "Hồ Chí Minh"},
    {"domain": "Backend", "title": "Node.js Backend Developer", "skills": ["Node.js", "Express", "MongoDB", "TypeScript", "Redis"], "exp": 2.0, "city": "Tất cả"},
    {"domain": "Backend", "title": ".NET Software Engineer", "skills": [".NET", "C#", "SQL Server", "Entity Framework"], "exp": 3.0, "city": "Đà Nẵng"},
    {"domain": "Backend", "title": "Golang Backend Engineer", "skills": ["Golang", "gRPC", "Docker", "Kubernetes", "PostgreSQL"], "exp": 2.5, "city": "Hồ Chí Minh"},
    
    # Phân hệ Frontend & Mobile (5 profiles)
    {"domain": "Frontend/Mobile", "title": "Frontend React Developer", "skills": ["React", "JavaScript", "TypeScript", "HTML5", "CSS3", "Redux"], "exp": 2.0, "city": "Hà Nội"},
    {"domain": "Frontend/Mobile", "title": "Vue.js Frontend Engineer", "skills": ["Vue", "Nuxt.js", "JavaScript", "Tailwind CSS"], "exp": 1.5, "city": "Hồ Chí Minh"},
    {"domain": "Frontend/Mobile", "title": "Mobile Flutter Developer", "skills": ["Flutter", "Dart", "Firebase", "RESTful API"], "exp": 2.0, "city": "Tất cả"},
    {"domain": "Frontend/Mobile", "title": "iOS Swift Developer", "skills": ["iOS", "Swift", "Xcode", "Objective-C"], "exp": 3.0, "city": "Hà Nội"},
    {"domain": "Frontend/Mobile", "title": "Android Kotlin Developer", "skills": ["Android", "Kotlin", "Java", "Coroutines"], "exp": 2.0, "city": "Hồ Chí Minh"},
    
    # Phân hệ Data & AI (5 profiles)
    {"domain": "Data/AI", "title": "Data Analyst", "skills": ["SQL", "Python", "Power BI", "Tableau", "Excel"], "exp": 1.5, "city": "Hà Nội"},
    {"domain": "Data/AI", "title": "Data Engineer", "skills": ["Python", "SQL", "Spark", "Kafka", "Hadoop", "Airflow"], "exp": 3.0, "city": "Hồ Chí Minh"},
    {"domain": "Data/AI", "title": "Machine Learning Engineer", "skills": ["Python", "PyTorch", "TensorFlow", "Scikit-Learn", "Docker", "MLOps"], "exp": 2.0, "city": "Tất cả"},
    {"domain": "Data/AI", "title": "AI NLP Engineer", "skills": ["Python", "NLP", "LLM", "Hugging Face", "LangChain"], "exp": 2.0, "city": "Hà Nội"},
    {"domain": "Data/AI", "title": "BI Developer", "skills": ["Power BI", "DAX", "SQL Server", "ETL", "Data Warehouse"], "exp": 2.5, "city": "Hồ Chí Minh"},
    
    # Phân hệ DevOps & Cloud & Security (5 profiles)
    {"domain": "DevOps/Cloud", "title": "DevOps Engineer", "skills": ["Docker", "Kubernetes", "CI/CD", "AWS", "Terraform", "Linux"], "exp": 3.0, "city": "Hà Nội"},
    {"domain": "DevOps/Cloud", "title": "Cloud AWS Architect", "skills": ["AWS", "Terraform", "Linux", "CloudFormation", "Docker"], "exp": 4.0, "city": "Hồ Chí Minh"},
    {"domain": "DevOps/Cloud", "title": "Cyber Security Specialist", "skills": ["Network Security", "Penetration Testing", "SOC", "Linux"], "exp": 3.0, "city": "Hà Nội"},
    {"domain": "DevOps/Cloud", "title": "System Administrator Linux", "skills": ["Linux", "Bash", "Shell", "Networking", "Docker"], "exp": 2.0, "city": "Đà Nẵng"},
    {"domain": "DevOps/Cloud", "title": "Site Reliability Engineer (SRE)", "skills": ["Kubernetes", "Prometheus", "Grafana", "Python", "Golang"], "exp": 3.5, "city": "Tất cả"},
    
    # Phân hệ Software Testing / QA (5 profiles)
    {"domain": "QA/QC", "title": "QA Automation Engineer", "skills": ["Selenium", "Python", "Java", "Postman", "CI/CD"], "exp": 2.0, "city": "Hà Nội"},
    {"domain": "QA/QC", "title": "Manual QC Tester", "skills": ["Manual Testing", "Test Case", "Jira", "Postman", "SQL"], "exp": 1.5, "city": "Hồ Chí Minh"},
    {"domain": "QA/QC", "title": "QA Lead", "skills": ["Test Plan", "Automation Testing", "Jira", "Performance Testing"], "exp": 4.0, "city": "Hà Nội"},
    {"domain": "QA/QC", "title": "Mobile App Tester", "skills": ["Appium", "Mobile Testing", "Postman", "Jira"], "exp": 2.0, "city": "Hồ Chí Minh"},
    {"domain": "QA/QC", "title": "Performance QA Engineer", "skills": ["JMeter", "Performance Testing", "LoadRunner", "SQL"], "exp": 3.0, "city": "Tất cả"}
]

# 2. Tiến hành tính toán các chỉ số với K = 5 và K = 10
K_VALS = [5, 10]
evaluation_results = []
domain_metrics = {d: {'scr': [], 'ild': [], 'precision': [], 'sims': []} for d in ["Backend", "Frontend/Mobile", "Data/AI", "DevOps/Cloud", "QA/QC"]}
all_recommended_jobs = set()
all_sim_scores = []

for p in test_benchmark_profiles:
    # Lấy top 10 gợi ý
    df_rec = recommend_jobs(
        target_title=p["title"],
        skills=p["skills"],
        user_exp=p["exp"],
        preferred_city=p["city"],
        top_n=10
    )
    
    if df_rec.empty:
        continue
        
    p_skills_lower = set(s.lower() for s in p["skills"])
    domain = p["domain"]
    
    # Duyệt K = 5
    top5 = df_rec.head(5)
    rec_indices = []
    
    for _, r in top5.iterrows():
        # Match back index in jobs
        m = jobs[jobs['title'] == r['Tiêu đề']]
        if not m.empty:
            rec_indices.append(m.index[0])
            all_recommended_jobs.add(m.index[0])
            
    # Điểm tương đồng
    scores5 = top5['Độ phù hợp (%)'].values / 100.0
    all_sim_scores.extend(scores5)
    domain_metrics[domain]['sims'].extend(scores5)
    
    # 1. Skill Coverage Rate (SCR@5)
    scr_list = []
    for _, r in top5.iterrows():
        txt = r['Độ phủ kỹ năng (%)'] # dạng '80.0% (4/5)'
        try:
            val = float(txt.split('%')[0]) / 100.0
            scr_list.append(val)
        except:
            scr_list.append(0.5)
    avg_scr = np.mean(scr_list)
    domain_metrics[domain]['scr'].append(avg_scr)
    
    # 2. Intra-List Diversity (ILD@5)
    if len(rec_indices) > 1:
        item_vecs = tfidf_matrix[rec_indices].toarray()
        cos_sim_mat = cosine_similarity(item_vecs)
        n = len(rec_indices)
        dists = [1.0 - cos_sim_mat[i, j] for i in range(n) for j in range(i + 1, n)]
        ild = np.mean(dists)
    else:
        ild = 0.0
    domain_metrics[domain]['ild'].append(ild)
    
    # 3. Domain Precision@5
    matched_domain = 0
    for _, r in top5.iterrows():
        title_low = str(r['Tiêu đề']).lower()
        skills_matched = str(r['Kỹ năng đã có (Điểm mạnh)']).lower()
        if any(sk in title_low or sk in skills_matched for sk in p_skills_lower):
            matched_domain += 1
    prec5 = matched_domain / len(top5) if len(top5) > 0 else 0.0
    domain_metrics[domain]['precision'].append(prec5)

# 3. Tổng hợp Bảng Chỉ số Đo lường Hệ thống
overall_scr = np.mean([np.mean(domain_metrics[d]['scr']) for d in domain_metrics])
overall_ild = np.mean([np.mean(domain_metrics[d]['ild']) for d in domain_metrics])
overall_prec = np.mean([np.mean(domain_metrics[d]['precision']) for d in domain_metrics])
catalog_cov = len(all_recommended_jobs) / len(jobs) * 100.0

print("=" * 80)
print("           BẢNG CHỈ SỐ ĐÁNH GIÁ HIỆU NĂNG CONTENT-BASED FILTERING")
print("=" * 80)
metrics_summary = pd.DataFrame([
    {
        'Chỉ số (Metric)': 'Skill Coverage Rate (SCR@5)',
        'Giá trị': f"{overall_scr * 100:.2f}%",
        'Ý nghĩa': 'Mức độ đáp ứng kỹ năng trung bình của ứng viên cho các việc được gợi ý'
    },
    {
        'Chỉ số (Metric)': 'Intra-List Diversity (ILD@5)',
        'Giá trị': f"{overall_ild:.4f}",
        'Ý nghĩa': 'Độ đa dạng danh sách gợi ý (thang đo 0 đến 1, càng cao càng đa dạng)'
    },
    {
        'Chỉ số (Metric)': 'Domain Precision (Precision@5)',
        'Giá trị': f"{overall_prec * 100:.2f}%",
        'Ý nghĩa': 'Tỷ lệ việc làm gợi ý rơi vào đúng phân hệ chuyên môn của ứng viên'
    },
    {
        'Chỉ số (Metric)': 'Catalog Coverage',
        'Giá trị': f"{catalog_cov:.2f}% ({len(all_recommended_jobs)}/{len(jobs)} jobs)",
        'Ý nghĩa': 'Tỷ lệ tổng số tin việc làm trong kho dữ liệu được hệ thống bao phủ qua 25 profiles'
    },
    {
        'Chỉ số (Metric)': 'Cosine Similarity (Mean ± Std)',
        'Giá trị': f"{np.mean(all_sim_scores):.4f} ± {np.std(all_sim_scores):.4f}",
        'Ý nghĩa': 'Phổ điểm tương đồng Cosine trung bình (Min: ' + f"{np.min(all_sim_scores):.4f}" + ', Max: ' + f"{np.max(all_sim_scores):.4f}" + ')'
    }
])
display(metrics_summary)

print("\n" + "=" * 80)
print("           PHÂN TÍCH CHI TIẾT THEO TỪNG PHÂN HỆ IT CHUYÊN MÔN")
print("=" * 80)
domain_summary = []
for d in domain_metrics:
    domain_summary.append({
        'Phân hệ IT': d,
        'Độ phủ kỹ năng (SCR@5)': f"{np.mean(domain_metrics[d]['scr']) * 100:.1f}%",
        'Độ đa dạng (ILD@5)': f"{np.mean(domain_metrics[d]['ild']):.4f}",
        'Độ chính xác Domain (Precision@5)': f"{np.mean(domain_metrics[d]['precision']) * 100:.1f}%",
        'Cosine Sim TB': f"{np.mean(domain_metrics[d]['sims']):.4f}"
    })
df_domain = pd.DataFrame(domain_summary)
display(df_domain)


           BẢNG CHỈ SỐ ĐÁNH GIÁ HIỆU NĂNG CONTENT-BASED FILTERING


,Chỉ số (Metric),Giá trị,Ý nghĩa
0,Skill Coverage Rate (SCR@5),30.90%,Mức độ đáp ứng kỹ năng trung bình của ứng viên...
1,Intra-List Diversity (ILD@5),0.7003,"Độ đa dạng danh sách gợi ý (thang đo 0 đến 1, ..."
2,Domain Precision (Precision@5),88.80%,Tỷ lệ việc làm gợi ý rơi vào đúng phân hệ chuy...
3,Catalog Coverage,2.87% (113/3938 jobs),Tỷ lệ tổng số tin việc làm trong kho dữ liệu đ...
4,Cosine Similarity (Mean ± Std),0.3203 ± 0.1209,Phổ điểm tương đồng Cosine trung bình (Min: 0....



           PHÂN TÍCH CHI TIẾT THEO TỪNG PHÂN HỆ IT CHUYÊN MÔN


,Phân hệ IT,Độ phủ kỹ năng (SCR@5),Độ đa dạng (ILD@5),Độ chính xác Domain (Precision@5),Cosine Sim TB
0,Backend,23.3%,0.7084,84.0%,0.2964
1,Frontend/Mobile,34.4%,0.6781,100.0%,0.3830
2,Data/AI,37.4%,0.6755,100.0%,0.3548
3,DevOps/Cloud,34.1%,0.6767,80.0%,0.2714
4,QA/QC,25.4%,0.7628,80.0%,0.2960
